<a href="https://colab.research.google.com/github/bah862696-coder/DI-Bootcamp/blob/master/Exercises_XP_VDB_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Vector Databases and RAG
Use this guided notebook and fill each TODO before running cells.

## What you'll learn
- Vector search strategies (KNN, ANN) and evaluation.
- Vector database utility (similarity search, RAG).
- Differences between vector DBs, libraries, and plugins.
- Best practices for vector store usage and performance.
- How LMs use context; embedding generation and storage.
- Querying vector stores and applying LMs for QA with retrieved context.

## What you'll build
A functional RAG pipeline with FAISS and ChromaDB, plus QA over retrieved context using a Hugging Face model.

## 0. Setup
Run the install cell once. If your platform needs system deps (e.g., libomp for FAISS), follow instructions in comments.

In [ ]:
%pip uninstall -y pydantic-core pydantic
%pip install -U "pydantic<2"
%pip install -U "faiss-cpu>=1.8.0" "chromadb==0.3.21"
%pip install -U "numpy<2" sentence-transformers transformers

In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from IPython.display import display
os.makedirs('cache', exist_ok=True)


## 🌟 Exercise 1 · Data loading and preparation

In [ ]:
data_path = 'labelled_newscatcher_dataset.csv'
pdf = pd.read_csv(data_path, sep=';')
if 'id' not in pdf.columns:
    pdf['id'] = range(len(pdf))  # TODO: replace with your own identifier logic if provided
display(pdf.head())
# TODO: create a manageable subset (e.g., first 1000 rows)
pdf_subset = pdf.head(1000)
pdf_subset[['id', 'title']].head()

## 🌟 Exercise 2 · Vectorization with Sentence Transformers

In [ ]:
def example_create_fn(idx: int, text: str) -> InputExample:
    return InputExample(guid=str(idx), texts=[text], label=0.0)

# Todo: create training examples from the subset data using the example_create_fn
faiss_train_examples = [example_create_fn(row.id, row.title) for row in pdf_subset.itertuples()]
faiss_train_examples[:2]

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
titles_list = pdf_subset['title'].tolist()
faiss_title_embedding = model.encode(titles_list, convert_to_numpy=True, show_progress_bar=True)
len(faiss_title_embedding), len(faiss_title_embedding[0])


## 🌟 Exercise 3 · FAISS indexing and search

In [ ]:
pdf_to_index = pdf_subset
id_index = pdf_to_index['id'].to_numpy().astype(np.int64)
content_encoded_normalized = faiss_title_embedding.astype('float32')
faiss.normalize_L2(content_encoded_normalized)
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(content_encoded_normalized.shape[1]))
index_content.add_with_ids(content_encoded_normalized, id_index)
index_content.ntotal


In [ ]:
def search_content(query: str, pdf_to_index: pd.DataFrame, k: int = 3):
    # TODO: encode the query using the sentence transformer model
    query_vector = model.encode(...)
    faiss.normalize_L2(query_vector)
    sims, ids = index_content.search(...)
    results = pdf_to_index[pdf_to_index['id'].isin(ids[0])].copy()
    results['similarities'] = sims[0]
    return results

display(search_content('animal', pdf_to_index, k=5))


## 🌟 Exercise 4 · ChromaDB collection and querying

In [ ]:
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))
collection_name = 'my_news'
if any(c.name == collection_name for c in chroma_client.list_collections()):
    chroma_client.delete_collection(name=collection_name)
# To-Do: create the collection and add documents
collection = chroma_client.create_collection(...)
collection.add(...)
results = collection.query(...)
print(json.dumps(results, indent=2))


## 🌟 Exercise 5 · Question answering with a Hugging Face model

In [ ]:
model_id = 'google/flan-t5-small'  # lightweight, better than tiny GPT-2 for QA
# To-Do: create the text2text-generation pipeline
pipe = pipeline(...)

question = "What's the latest news on space development?"
context_docs = results['documents'][0][:3]
context = ' '.join(context_docs)
prompt = f"Answer the question using only the context.\nContext: {context}\nQuestion: {question}\nAnswer:\n"
response = pipe(prompt)[0]['generated_text']
print(response)
